In [53]:
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator
from ta.trend import EMAIndicator, SMAIndicator

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest


from datetime import timedelta

In [54]:
# from pytz import timezone
# tz = timezone('Europe/London')
# # print(tz._utc_transition_times)
# for transition in tz._utc_transition_times:
#     print(transition)

In [55]:
# swing structure
def implement_market_structure_states_strict(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = 'NEUTRAL'

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH":
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "NEUTRAL"
                higher_low = None  # Reset des repères obsolètes
                lower_high = None

        elif current_state == "BEARISH":
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "NEUTRAL"
                higher_low = None
                lower_high = None  # Reset des repères obsolètes

        elif current_state == "NEUTRAL":
            # En zone neutre, on attend un NOUVEAU BOS pour définir la tendance
            # Entrée en Bullish : Clôture au-dessus du dernier sommet confirmé
            # il faut tjs un high and un low avant de former une structure bearish ou bullish
            if last_confirmed_high is not None and last_confirmed_low is not None and close_p > last_confirmed_high:
                current_state = "BULLISH"
                higher_low = (
                    last_confirmed_low  # On fige le dernier creux comme HL
                )

            # Entrée en Bearish : Clôture sous le dernier creux confirmé
            elif last_confirmed_high is not None and last_confirmed_low is not None and close_p < last_confirmed_low:
                current_state = "BEARISH"
                lower_high = (
                    last_confirmed_high  # On fige le dernier sommet comme LH
                )

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            if current_state == "BEARISH" and lower_high > last_confirmed_high:
                lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            if current_state == "BULLISH" and higher_low < last_confirmed_low:
                higher_low = last_confirmed_low

    df["market_state"] = states
    return df


In [56]:
# swing structure direct Bullish to Bearish
def implement_market_structure_states_direct(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = None

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH" or current_state is None:
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "BEARISH"

        elif current_state == "BEARISH" or current_state is None:
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "BULLISH"

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            higher_low = last_confirmed_low


    df["market_state"] = states
    return df


In [80]:
# variables

structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2026-1-1 00:00', date_format)
end_date = datetime.strptime('2026-9-16 00:00', date_format)
# start_date = '2022-01-1 00:00'
# end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

In [81]:
df_d1_raw = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"./Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"./Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-10  129.605  130.065  128.995  129.255   172229
2010-09-13  129.565  130.115  128.780  129.090   215859
2010-09-14  129.105  129.430  127.640  129.015   226170
2010-09-15  129.015  134.070  128.555  133.975   252317
2010-09-16  133.975  134.260  132.970  134.150   222443
...             ...      ...      ...      ...      ...
2026-09-18  208.323  211.276  208.233  210.095   364707
2026-09-21  210.296  210.762  209.676  210.321   243731
2026-09-22  210.321  210.892  209.513  210.080   333200
2026-09-23  210.080  210.176  209.422  209.564   273936
2026-09-24  209.566  210.105  208.768  209.848   199988

[4183 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-10 04:00:00  129.605  129.630  128.995  129.385    25539
2010-09-10 08:00:00  129.380  129.985  129.275  129.

In [82]:
# process D1 data

df_d1 = df_d1_raw.loc[start_date - timedelta(days=30) : end_date]

df_d1 = implement_market_structure_states_strict(df_d1, n=structure_d1_window)
df_d1["state_changed"] = df_d1["market_state"] != df_d1[
    "market_state"
].shift(1)
print(df_d1[df_d1["state_changed"]][["Close", "market_state"]])
df_d1 = df_d1.loc[start_date : end_date]

              Close market_state
Date                            
2025-12-02  205.840      NEUTRAL
2025-12-19  210.975      BULLISH
2026-02-10  210.561      NEUTRAL
2026-02-11  208.614      BEARISH
2026-02-24  210.224      NEUTRAL
2026-02-25  211.860      BULLISH
2026-03-30  210.598      NEUTRAL
2026-03-31  209.933      BEARISH
2026-04-09  213.599      NEUTRAL
2026-04-10  214.376      BULLISH
2026-04-30  213.107      NEUTRAL
2026-05-25  214.592      BULLISH
2026-06-18  212.961      NEUTRAL
2026-06-29  214.706      BULLISH
2026-07-30  215.005      NEUTRAL
2026-07-31  212.145      BEARISH


In [60]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]

indicator_ema_20 = EMAIndicator(df_h4['Close'], window=20)
df_h4['EMA_20'] = indicator_ema_20.ema_indicator()

indicator_ema_50 = EMAIndicator(df_h4['Close'], window=50)
df_h4['EMA_50'] = indicator_ema_50.ema_indicator()

df_h4 = df_h4.loc[start_date : end_date]

h4_d1_direction_list = []
for i in range(len(df_d1)):
    row = df_d1.iloc[i]
    start_date_daily = row.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h4.loc[f'{start_date_daily}' : f'{end_date_daily}']

    for index2, row2 in df_by_day.iterrows():
        h4_d1_direction_list.append(df_d1.iloc[i-1]['market_state'])

df_h4['market_state_d1'] = h4_d1_direction_list

df_h4

,Open,High,Low,Close,Tickvol,EMA_20,EMA_50,market_state_d1
Date,,,,,,,,
2024-01-01 20:00:00,179.168,179.414,179.143,179.271,828,180.300720,180.788473,NEUTRAL
2024-01-02 00:00:00,179.272,180.218,179.261,179.927,18309,180.265128,180.754690,NEUTRAL
2024-01-02 04:00:00,179.927,180.207,179.894,180.166,13083,180.255687,180.731604,NEUTRAL
2024-01-02 08:00:00,180.162,180.646,179.792,179.821,38466,180.214288,180.695894,NEUTRAL
2024-01-02 12:00:00,179.818,179.944,178.740,178.749,48038,180.074737,180.619545,NEUTRAL
...,...,...,...,...,...,...,...,...
2024-12-27 08:00:00,197.569,197.819,197.340,197.702,28385,197.217998,196.407272,BEARISH
2024-12-27 12:00:00,197.703,198.500,197.522,198.363,46193,197.327045,196.483968,BEARISH
2024-12-27 16:00:00,198.366,198.500,198.017,198.354,64763,197.424851,196.557302,BEARISH


In [61]:
# process m15 data
df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]
df_m15 = implement_market_structure_states_direct(df_m15, n=structure_m15_window)

m15_tf_ema_20_list = []
m15_tf_ema_50_list = []
m15_daily_direction_list = []

df_m15 = df_m15.loc[start_date : end_date]
for i in range(len(df_h4)):
    row = df_h4.iloc[i]
    start_date_h4 = row.name
    end_date_h4 = start_date_h4 + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_h4}' : f'{end_date_h4}']

    for index2, row2 in df_by_day.iterrows():
        m15_tf_ema_50_list.append(df_h4.iloc[i-1]['EMA_50'])
        m15_tf_ema_20_list.append(df_h4.iloc[i-1]['EMA_20'])
        # affect current market states of H4 candle because the result is market state of previous D1 candle
        m15_daily_direction_list.append(df_h4.iloc[i]['market_state_d1'])

df_m15['EMA_50'] = m15_tf_ema_50_list
df_m15['EMA_20'] = m15_tf_ema_20_list
df_m15['market_state_d1'] = m15_daily_direction_list

indicator_atr_14 = AverageTrueRange(df_m15['High'], df_m15['Low'], df_m15['Close'], window=14)
df_m15['ATR'] = indicator_atr_14._atr

df_m15

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,EMA_50,EMA_20,market_state_d1,ATR
Date,,,,,,,,,,,,
2024-01-01 23:00:00,179.168,179.221,179.143,179.212,21,False,False,NaN,196.704214,197.614170,NEUTRAL,0.000000
2024-01-01 23:15:00,179.212,179.344,179.151,179.281,209,False,False,NaN,196.704214,197.614170,NEUTRAL,0.000000
2024-01-01 23:30:00,179.283,179.414,179.181,179.313,374,True,False,NaN,196.704214,197.614170,NEUTRAL,0.000000
2024-01-01 23:45:00,179.319,179.329,179.244,179.271,224,False,False,NaN,196.704214,197.614170,NEUTRAL,0.000000
2024-01-02 00:00:00,179.272,179.401,179.261,179.292,936,False,False,NaN,180.788473,180.300720,NEUTRAL,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-27 22:00:00,198.634,198.649,198.588,198.608,1479,True,False,BULLISH,196.557302,197.424851,BEARISH,0.133164
2024-12-27 22:15:00,198.608,198.639,198.598,198.598,872,False,False,BULLISH,196.557302,197.424851,BEARISH,0.126581
2024-12-27 22:30:00,198.598,198.624,198.521,198.543,671,False,False,BULLISH,196.557302,197.424851,BEARISH,0.124897


In [62]:
df_m15.describe()

,Open,High,Low,Close,Tickvol,EMA_50,EMA_20,ATR
count,24464.000000,24464.000000,24464.000000,24464.000000,24464.000000,24464.000000,24464.000000,24464.000000
mean,193.492433,193.571574,193.411021,193.493719,2177.937541,193.235020,193.378504,0.161106
std,5.512033,5.505183,5.517187,5.511345,1590.282397,5.414466,5.491509,0.083397
min,178.745000,178.913000,178.731000,178.749000,2.000000,180.423062,179.848247,0.000000
25%,189.516500,189.607750,189.440000,189.517000,1089.750000,189.625829,189.680148,0.108279
50%,192.431000,192.522500,192.352500,192.435000,1817.000000,191.966082,192.116953,0.142372
75%,197.412000,197.493000,197.324000,197.412250,2816.250000,197.189082,197.420890,0.189823
max,208.078000,208.106000,208.030000,208.080000,13374.000000,205.778639,206.889329,0.855330


In [63]:
def is_all_candle_above(df, threshold):
    print(len(df))
    for candle in df.iterrows():
        if candle[1]['Low'] < candle[1]['EMA_50']:
            return False

    return True

def is_all_candle_below(df, threshold):
    for candle in  df.iterrows():
        if candle[1]['High'] > candle[1]['EMA_50']:
            return False

    return True

In [64]:
#using simple entry model 
def apply_total_signal(df, ema50_window = 40):
    # Initialize the 'TotalSignal' column
    df_signal_list = []
    start_trade_hour = 5
    stop_trade_hour = 20

    for i in range(0, len(df)):
        signal = 0
        c_time_trade = start_trade_hour < df.index[i].hour and df.index[i].hour < stop_trade_hour

        if c_time_trade:
            # d1 trending conditions
            c_long_daily_direction = df['market_state_d1'].iloc[i-1] == 'BULLISH'
            c_40_candles_above_ema50 = (df['Low'].iloc[i-ema50_window:i-1] > df['EMA_50'].iloc[i-1]).all()
            # c_40_candles_above_ema50 = is_all_candle_above(df[i-ema50_window:i-1], df['EMA_50'].iloc[i-1])

            # h4 ema conditions
            c_h4_ema20_above_ema50 = df['EMA_20'].iloc[i-1] > df['EMA_50'].iloc[i-1]
            c_h4_low_below_ema20 = df['Low'].iloc[i-1] < df['EMA_20'].iloc[i-1]
            c_h4_close_above_ema50 = df['Close'].iloc[i-1] > df['EMA_50'].iloc[i-1]

            # Current candle conditions
            closes_above_prev_high = df['Close'].iloc[i] > df['High'].iloc[i-1]

            # Combine conditions
            if (c_long_daily_direction and
                c_h4_ema20_above_ema50 and
                c_h4_low_below_ema20 and
                c_h4_close_above_ema50 and
                closes_above_prev_high and 
                c_40_candles_above_ema50):
                signal = 2

            # d1 trending conditions
            c_short_daily_direction = df['market_state_d1'].iloc[i-1] == 'BEARISH'
            c_40_candles_below_ema50 = (df['High'].iloc[i-ema50_window:i-1] < df['EMA_50'].iloc[i-1]).all()
            # c_40_candles_below_ema50 = is_all_candle_below(df[i-ema50_window:i-1], df['EMA_50'].iloc[i-1])

            # h4 ema conditions
            c_h4_ema20_below_ema50 = df['EMA_20'].iloc[i-1] < df['EMA_50'].iloc[i-1]
            c_h4_low_above_ema20 = df['Low'].iloc[i-1] > df['EMA_20'].iloc[i-1]
            c_h4_close_below_ema50 = df['Close'].iloc[i-1] < df['EMA_50'].iloc[i-1]

            # Current candle conditions
            closes_below_prev_low = df['Close'].iloc[i] < df['Low'].iloc[i-1]

            # Combine conditions
            if (c_short_daily_direction and
                c_h4_ema20_below_ema50 and
                c_h4_low_above_ema20 and
                c_h4_close_below_ema50 and
                closes_below_prev_low and
                c_40_candles_below_ema50):
                signal = 1

        df_signal_list.append(signal)

    return df_signal_list

df_m15['TotalSignal'] = apply_total_signal(df=df_m15, ema50_window=40)

print(df_m15)

                        Open     High      Low    Close  Tickvol  \
Date                                                               
2024-01-01 23:00:00  179.168  179.221  179.143  179.212       21   
2024-01-01 23:15:00  179.212  179.344  179.151  179.281      209   
2024-01-01 23:30:00  179.283  179.414  179.181  179.313      374   
2024-01-01 23:45:00  179.319  179.329  179.244  179.271      224   
2024-01-02 00:00:00  179.272  179.401  179.261  179.292      936   
...                      ...      ...      ...      ...      ...   
2024-12-27 22:00:00  198.634  198.649  198.588  198.608     1479   
2024-12-27 22:15:00  198.608  198.639  198.598  198.598      872   
2024-12-27 22:30:00  198.598  198.624  198.521  198.543      671   
2024-12-27 22:45:00  198.542  198.551  198.432  198.448     1176   
2024-12-30 00:00:00  198.254  198.593  198.250  198.559      892   

                     is_swing_high  is_swing_low market_state      EMA_50  \
Date                                  

In [65]:
# using m15 market structure for entry
def apply_total_signal(df, ema50_window = 40):
    # Initialize the 'TotalSignal' column
    df_signal_list = []
    start_trade_hour = 5
    stop_trade_hour = 20
    current_day = None
    high_tf_condition_long_satisfied = False
    high_tf_condition_long_candle = None
    high_tf_condition_short_satisfied = False
    high_tf_condition_short_candle = None

    # we want market structure to switch from bearish to bullish when htf conditions are met
    # if market structure is already bullish we need to wait
    previous_market_structure = None

    for i in range(0, len(df)):
        current_bar_date = df.index[i]

        if current_day != current_bar_date.date() or current_bar_date.time().hour >= stop_trade_hour:
            current_day = current_bar_date.date()
            high_tf_condition_long_satisfied = False
            high_tf_condition_short_satisfied = False

        signal = 0
        c_time_trade = start_trade_hour < current_bar_date.hour and current_bar_date.hour < stop_trade_hour



        if c_time_trade:
            # d1 trending conditions
            c_long_daily_direction = df['market_state_d1'].iloc[i-1] == 'BULLISH'
            # c_40_candles_above_ema50 = is_all_candle_above(df[i-ema50_window:i-1], df['EMA_50'].iloc[i-1])

            # h4 ema conditions
            c_h4_ema20_above_ema50 = df['EMA_20'].iloc[i-1] > df['EMA_50'].iloc[i-1]
            c_h4_low_below_ema20 = df['Low'].iloc[i-1] < df['EMA_20'].iloc[i-1]

            # if after all conditions are met then one candle is below ema50, wait till new setup
            c_40_candles_above_ema50 = (df['Low'].iloc[i-ema50_window:i-1] > df['EMA_50'].iloc[i-1]).all()
            
            # m15 market structure shift to bullish
            c_m15_ms_bullish = df['market_state'].iloc[i-1] == 'BULLISH'

            # Combine conditions
            if (c_long_daily_direction and
                c_h4_ema20_above_ema50 and
                c_h4_low_below_ema20 and
                high_tf_condition_long_satisfied == False):
                high_tf_condition_long_satisfied = True
                high_tf_condition_long_candle = current_bar_date
                previous_market_structure = df['market_state'].iloc[i-1]

            if not c_40_candles_above_ema50:
                high_tf_condition_long_satisfied = False
                high_tf_condition_long_candle = None

            if (high_tf_condition_long_satisfied == True and
                c_m15_ms_bullish and
                high_tf_condition_long_candle < current_bar_date and
                previous_market_structure == 'BEARISH'):
                # print(f"high_tf_condition_long_candle = {high_tf_condition_long_candle} & current_bar_date = {current_bar_date}")
                signal = 2

            # d1 trending conditions
            c_short_daily_direction = df['market_state_d1'].iloc[i-1] == 'BEARISH'
            # c_40_candles_below_ema50 = is_all_candle_below(df[i-ema50_window:i-1], df['EMA_50'].iloc[i-1])

            # h4 ema conditions
            c_h4_ema20_below_ema50 = df['EMA_20'].iloc[i-1] < df['EMA_50'].iloc[i-1]
            c_h4_low_above_ema20 = df['Low'].iloc[i-1] > df['EMA_20'].iloc[i-1]

            # if after all conditions are met then one candle is above ema50, wait till new setup
            c_40_candles_below_ema50 = (df['High'].iloc[i-ema50_window:i-1] < df['EMA_50'].iloc[i-1]).all()

            # m15 market structure shift to bullish
            c_m15_ms_bearish = df['market_state'].iloc[i-1] == 'BEARISH'
            
            # Combine conditions
            if (c_short_daily_direction and
                c_h4_ema20_below_ema50 and
                c_h4_low_above_ema20 and
                high_tf_condition_short_satisfied):
                high_tf_condition_short_satisfied = True
                high_tf_condition_short_candle = current_bar_date
                previous_market_structure = df['market_state'].iloc[i-1]

            if not c_40_candles_below_ema50:
                high_tf_condition_short_satisfied = False
                high_tf_condition_short_candle = None
    
            if (high_tf_condition_short_satisfied == True and
                c_m15_ms_bearish and
                high_tf_condition_short_candle < current_bar_date and
                previous_market_structure == 'BULLISH'):
                # print(f"high_tf_condition_long_candle = {high_tf_condition_long_candle} & current_bar_date = {current_bar_date}")
                signal = 1

            if previous_market_structure != df['market_state'].iloc[i-1]:
                previous_market_structure = df['market_state'].iloc[i-1]

        df_signal_list.append(signal)

    return df_signal_list

df_m15['TotalSignal'] = apply_total_signal(df=df_m15, ema50_window=40)

print(df_m15)

                        Open     High      Low    Close  Tickvol  \
Date                                                               
2024-01-01 23:00:00  179.168  179.221  179.143  179.212       21   
2024-01-01 23:15:00  179.212  179.344  179.151  179.281      209   
2024-01-01 23:30:00  179.283  179.414  179.181  179.313      374   
2024-01-01 23:45:00  179.319  179.329  179.244  179.271      224   
2024-01-02 00:00:00  179.272  179.401  179.261  179.292      936   
...                      ...      ...      ...      ...      ...   
2024-12-27 22:00:00  198.634  198.649  198.588  198.608     1479   
2024-12-27 22:15:00  198.608  198.639  198.598  198.598      872   
2024-12-27 22:30:00  198.598  198.624  198.521  198.543      671   
2024-12-27 22:45:00  198.542  198.551  198.432  198.448     1176   
2024-12-30 00:00:00  198.254  198.593  198.250  198.559      892   

                     is_swing_high  is_swing_low market_state      EMA_50  \
Date                                  

In [66]:
len(df_m15[df_m15.TotalSignal != 0])

35

In [67]:
def pointpos(x):
    if x['TotalSignal']==2:
        return x['Low']-1e-4
    elif x['TotalSignal']==1:
        return x['High']+1e-4
    else:
        return np.nan

df_m15['pointpos'] = df_m15.apply(lambda row: pointpos(row), axis=1)

In [68]:
# Affichage des transitions d'états D1
df_m15["state_changed"] = df_m15["market_state_d1"] != df_m15[
    "market_state_d1"
].shift(1)
df_state_changed = df_m15[df_m15["state_changed"]][["Close", "market_state_d1"]]

def martket_structure_bullish_pos(x):
    for row in df_state_changed.iterrows():
        if x.name == row[0]:
            if row[1]['market_state_d1'] == 'BULLISH':
                return x['Low']-1e-4

def martket_structure_bearish_pos(x):
    for row in df_state_changed.iterrows():
        if x.name == row[0]:
            if row[1]['market_state_d1'] == 'BEARISH':
                return x['High']+1e-4

def martket_structure_neutral_pos(x):
    for row in df_state_changed.iterrows():
        if x.name == row[0]:
            if row[1]['market_state_d1'] == 'NEUTRAL':
                return x['High']+1e-4

df_m15['mk_bullish_pos'] = df_m15.apply(lambda row: martket_structure_bullish_pos(row), axis=1)
df_m15['mk_bearish_pos'] = df_m15.apply(lambda row: martket_structure_bearish_pos(row), axis=1)
df_m15['mk_neutral_pos'] = df_m15.apply(lambda row: martket_structure_neutral_pos(row), axis=1)

In [69]:
# Affichage des transitions d'états m15
df_m15["state_changed"] = df_m15["market_state"] != df_m15[
    "market_state"
].shift(1)
df_state_changed = df_m15[df_m15["state_changed"]][["Close", "market_state"]]

mk_bullish_pos_list = []
mk_bearish_pos_list = []

for i in range(len(df_m15)):
    previous_candle = df_m15.iloc[i-1]
    candle = df_m15.iloc[i]
    if candle['market_state'] == 'BULLISH' and previous_candle['market_state'] != 'BULLISH':
        mk_bullish_pos_list.append(candle['Low'] - 1e-4)
        mk_bearish_pos_list.append(np.nan)
        continue        

    if candle['market_state'] == 'BEARISH' and previous_candle['market_state'] != 'BEARISH':
        mk_bullish_pos_list.append(np.nan)
        mk_bearish_pos_list.append(candle['High'] + 1e-4)
        continue

    mk_bullish_pos_list.append(np.nan)
    mk_bearish_pos_list.append(np.nan)

df_m15['mk_bullish_pos'] = mk_bullish_pos_list
df_m15['mk_bearish_pos'] = mk_bearish_pos_list

In [70]:
def swing_point_pos(x):
    if x['is_swing_low']==True:
        return x['Low']-1e-4
    elif x['is_swing_high']==True:
        return x['High']+1e-4
    else:
        return np.nan

df_m15['swing_point'] = df_m15.apply(lambda row: swing_point_pos(row), axis=1)

In [ ]:
start_date_chart = '2026-7-1 00:00:00'
end_date_chart = '2026-7-31 00:00:00'
dfpl = df_m15.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")
number_trades = len(dfpl[dfpl.TotalSignal != 0])

print(number_trades)
# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

# Add Bollinger Bands, EMA lines on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['EMA_50'],
                         line=dict(color='blue', width=1),
                         name="EMA_50"),
              row=1, col=1)
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['EMA_20'],
                         line=dict(color='yellow', width=1),
                         name="EMA_20"),
              row=1, col=1)

# Add markers for trade entry points on the same subplot
# fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['swing_point'], mode="markers",
#                          marker=dict(size=6, color="black"),
#                          name="swing point"),
#               row=1, col=1)

# Add markers for swing point m15
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['pointpos'], mode="markers",
                         marker=dict(size=12, color="MediumPurple"),
                         name="entry"),
              row=1, col=1)

# Add markers for market structure daily = bullish
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['mk_bullish_pos'], mode="markers",
                         marker=dict(size=8, color="green"),
                         name="bullish"),
              row=1, col=1)

# Add markers for market structure daily = bearish
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['mk_bearish_pos'], mode="markers",
                         marker=dict(size=8, color="red"),
                         name="bearish"),
              row=1, col=1)

# Add markers for market structure daily = neutral
# fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['mk_neutral_pos'], mode="markers",
#                          marker=dict(size=12, color="yellow"),
#                          name="neutral"),
#               row=1, col=1)

fig.update_layout(width=1200, height=1000, sliders=[])
fig.show()

0


In [72]:
# simple sl tp management
dfopt = df_m15.loc['2025-01-1 00:00:00' : '2025-12-30 00:00:00']
# dfopt = df_m15
def SIGNAL():
    return dfopt.TotalSignal

class MyStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    last_swing_low = None
    last_swing_high = None
    
    def init(self):
        super().init()
        self.signal1 = self.I(SIGNAL)

    def next(self):
        super().next()

        # if len(self.trades) > 0:
        #     print(self.trades)
        if self.data['is_swing_low'][len(self.data['is_swing_low'])-3] == True:
            self.last_swing_low = self.data.Low[len(self.data['is_swing_low'])-3]

        if self.data['is_swing_high'][len(self.data['is_swing_high'])-3] == True:
            self.last_swing_high = self.data.High[len(self.data['is_swing_high'])-3]

        if len(self.trades)==0:
            if self.signal1==2:
                sl = abs(self.data.Close[-1] - self.data.Low[-1])
                # sl = self.data['ATR'][-1] * 1.5

                tp = self.TPcoef * sl
                c_sl_tp = sl < self.data['ATR'][-1] * 2.5 and sl > self.data['ATR'][-1] * 0.5
                position_size = int((self.risk_percent * self.equity) / sl)
                if c_sl_tp:
                    sl1 = self.data.Close[-1] - sl
                    tp1 = self.data.Close[-1] + tp
                    # print(f"go long date= {self.data.index[-1]} & sl1 = {sl1} & tp1 = {tp1}")
                    self.buy(sl=sl1, tp=tp1, size=position_size)

            if self.signal1==1:
                sl = abs(self.data.Close[-1] - self.data.High[-1])
                # sl = self.data['ATR'][-1] * 1.5

                tp = self.TPcoef * sl
                c_sl_tp = sl < self.data['ATR'][-1] * 2.5 and sl > self.data['ATR'][-1] * 0.5
                position_size = int((self.risk_percent * self.equity) / sl)
                if c_sl_tp:
                    sl1 = self.data.Close[-1] + sl
                    tp1 = self.data.Close[-1] - tp
                    # print(f"go short date= {self.data.index[-1]} & sl1 = {sl1} & tp1 = {tp1}")
                    self.sell(sl=sl1, tp=tp1, size=position_size)

In [78]:
# add some sl tp management
dfopt = df_m15.loc['2024-01-1 00:00:00' : '2024-12-30 00:00:00']
# dfopt = df_m15
def SIGNAL():
    return dfopt.TotalSignal

class MyStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    last_swing_low = None
    last_swing_high = None
    
    def init(self):
        super().init()
        self.signal1 = self.I(SIGNAL)

    def next(self):
        super().next()

        # if len(self.trades) > 0:
        #     print(self.trades)
        if self.data['is_swing_low'][len(self.data['is_swing_low'])-3] == True:
            self.last_swing_low = self.data.Low[len(self.data['is_swing_low'])-3]

        if self.data['is_swing_high'][len(self.data['is_swing_high'])-3] == True:
            self.last_swing_high = self.data.High[len(self.data['is_swing_high'])-3]

        if len(self.trades)==0:
            if self.signal1==2:
                sl = abs(self.data.Close[-1] - self.last_swing_low)
                # sl = self.data['ATR'][-1] * 1.5

                tp = self.TPcoef * sl
                c_sl_tp = sl < self.data['ATR'][-1] * 2.5 and sl > self.data['ATR'][-1] * 0.5
                position_size = int((self.risk_percent * self.equity) / sl)
                if c_sl_tp:
                    sl1 = self.data.Close[-1] - sl
                    tp1 = self.data.Close[-1] + tp
                    # print(f"go long date= {self.data.index[-1]} & sl1 = {sl1} & tp1 = {tp1}")
                    self.buy(sl=sl1, tp=tp1, size=position_size)

            if self.signal1==1:
                sl = abs(self.data.Close[-1] - self.last_swing_high)
                # sl = self.data['ATR'][-1] * 1.5

                tp = self.TPcoef * sl
                c_sl_tp = sl < self.data['ATR'][-1] * 2.5 and sl > self.data['ATR'][-1] * 0.5
                position_size = int((self.risk_percent * self.equity) / sl)
                if c_sl_tp:
                    sl1 = self.data.Close[-1] + sl
                    tp1 = self.data.Close[-1] - tp
                    # print(f"go short date= {self.data.index[-1]} & sl1 = {sl1} & tp1 = {tp1}")
                    self.sell(sl=sl1, tp=tp1, size=position_size)

In [79]:
bt = Backtest(dfopt, MyStrat, cash=10000, margin=0.02) #0.0002
stats = bt.run()

print(stats)

bt.plot()

Start                     2024-01-01 23:00:00
End                       2024-12-30 00:00:00
Duration                    363 days 01:00:00
Exposure Time [%]                     1.10366
Equity Final [$]                     9682.607
Equity Peak [$]                       10230.7
Return [%]                           -3.17393
Buy & Hold Return [%]                10.79559
Return (Ann.) [%]                    -3.07779
Volatility (Ann.) [%]                 5.00762
CAGR [%]                             -3.19293
Sharpe Ratio                         -0.61462
Sortino Ratio                        -0.90182
Calmar Ratio                         -0.37935
Alpha [%]                            -3.48558
Beta                                  0.02887
Max. Drawdown [%]                    -8.11327
Avg. Drawdown [%]                     -1.4178
Max. Drawdown Duration      341 days 08:00:00
Avg. Drawdown Duration       49 days 02:52:00
# Trades                                   15
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '1h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p2968', ...)